<img src='https://drive.google.com/uc?id=1tqYIvII8lJ_FnqE6ugS21n4s93kMwTLy' />

## Machine Learning
## School of Computing and Engineering, University of West London
## Massoud Zolgharni

# Convolutional Neural Networks

Train a simple Convolutional Neural Network (CNN) to classify [CIFAR images](https://www.cs.toronto.edu/~kriz/cifar.html).


### Import TensorFlow

In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt

### Download and prepare the CIFAR10 dataset


The CIFAR10 dataset contains 60,000 color images in 10 classes, with 6,000 images in each class. The dataset is divided into 50,000 training images and 10,000 testing images. The classes are mutually exclusive and there is no overlap between them.

In [ ]:
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

# Normalize pixel values to be between 0 and 1
train_images, test_images = train_images / 255.0, test_images / 255.0

### Verify the data

To verify that the dataset looks correct, let's plot the first 25 images from the training set and display the class name below each image.


In [ ]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i], cmap=plt.cm.binary)
    # The CIFAR labels happen to be arrays,
    # which is why you need the extra index
    plt.xlabel(class_names[train_labels[i][0]])
plt.show()

### Create the convolutional base

Below we define the convolutional base using a common pattern: a stack of [Conv2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D) and [MaxPooling2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/MaxPool2D) layers.

As input, our CNN model takes tensors of shape (image_height, image_width, color_channels), ignoring the batch size.

Remember, color_channels refers to (R,G,B). In this example, we will configure our CNN to process inputs of shape (32, 32, 3), which is the format of CIFAR images. We can do this by passing the argument `input_shape` to our first layer.


In [ ]:
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))

Let's display the architecture of our model so far.

In [ ]:
model.summary()

Above, you can see that the output of every Conv2D and MaxPooling2D layer is a 3D tensor of shape (height, width, channels). The width and height dimensions tend to shrink as you go deeper in the network. The number of output channels for each Conv2D layer is controlled by the first argument (e.g., 32 or 64). Typically,  as the width and height shrink, you can afford (computationally) to add more output channels in each Conv2D layer.

### Add Dense layers on top
To complete our model, you will feed the last output tensor from the convolutional base (of shape (4, 4, 64)) into one or more Dense layers to perform classification. Dense layers take vectors as input (which are 1D), while the current output is a 3D tensor. First, you will flatten (or unroll) the 3D output to 1D,  then add one or more Dense layers on top. CIFAR has 10 output classes, so you use a final Dense layer with 10 outputs and a softmax activation.

In [ ]:
model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(10))

Here's the complete architecture of our model.

In [ ]:
model.summary()

As you can see, our (4, 4, 64) outputs were flattened into vectors of shape (1024) before going through two Dense layers.

### Compile and train the model

In [ ]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

history = model.fit(train_images, train_labels, epochs=3,
                    validation_data=(test_images, test_labels))

### Evaluate the model

In [ ]:
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.5, 1])
plt.legend(loc='lower right')

In [ ]:
test_loss, test_acc = model.evaluate(test_images,  test_labels)

In [ ]:
print('Test accuracy:', test_acc)

### Plot confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import itertools
import numpy as np
plt.rcParams['figure.figsize'] = [10,7]

def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):

  if normalize:
      cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
      print("Normalized confusion matrix")
  else:
      print('Confusion matrix, without normalization')

  print(cm)

  plt.imshow(cm, interpolation='nearest', cmap=cmap)
  plt.title(title)
  plt.colorbar()
  tick_marks = np.arange(len(classes))
  plt.xticks(tick_marks, classes, rotation=45)
  plt.yticks(tick_marks, classes)

  fmt = '.2f' if normalize else 'd'
  thresh = cm.max() / 2.
  for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
      plt.text(j, i, format(cm[i, j], fmt),
               horizontalalignment="center",
               color="white" if cm[i, j] > thresh else "black")

  plt.tight_layout()
  plt.ylabel('True label')
  plt.xlabel('Predicted label')
  plt.show()


predicted_labels = model.predict(test_images).argmax(axis=1)
cm = confusion_matrix(test_labels, predicted_labels)
plot_confusion_matrix(cm, list(range(10)))

# Your Task
Do the following:
*   Train the your CNN model for a few more iterations (you can do this by changing the number of **epochs** in **model.fit**, then click on **Runtime** and **Run all**)
*   Try it for 10, 20, 50 epochs.
*   How does the acuuracy change? Discuss your observations.
*   If accuracy is improving, explain why. If accuracy is getting worse, why do you think that is?
*   Change the kernel size from (3, 3) to (5, 5) in your CNN model (for all layers.Conv2D layers)
*   Would that help to improve the accuracy?















In [ ]:
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(10))
model.summary()

model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

# Training for 10 epochs
history_10_epochs = model.fit(train_images, train_labels, epochs=10,
                              validation_data=(test_images, test_labels))

plt.figure(figsize=(4, 4))
plt.plot(history_10_epochs.history['accuracy'], label='accuracy')
plt.plot(history_10_epochs.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.5, 1])
plt.title('history_10_epochs')
plt.legend(loc='lower right')

test_loss, test_acc = model.evaluate(test_images,  test_labels)

print('Test accuracy:', test_acc)

In [ ]:
# Training for 20 epochs
history_20_epochs = model.fit(train_images, train_labels, epochs=20,
                              validation_data=(test_images, test_labels))

plt.figure(figsize=(4, 4))
plt.plot(history_20_epochs.history['accuracy'], label='accuracy')
plt.plot(history_20_epochs.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.5, 1])
plt.title('history_20_epochs')
plt.legend(loc='lower right')

test_loss, test_acc = model.evaluate(test_images,  test_labels)

print('Test accuracy:', test_acc)

In [ ]:
# Training for 50 epochs
history_50_epochs = model.fit(train_images, train_labels, epochs=50,
                              validation_data=(test_images, test_labels))

plt.figure(figsize=(4, 4))
plt.plot(history_50_epochs.history['accuracy'], label='accuracy')
plt.plot(history_50_epochs.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.5, 1])
plt.title('history_50_epochs')
plt.legend(loc='lower right')

test_loss, test_acc = model.evaluate(test_images,  test_labels)

print('Test accuracy:', test_acc)

In [ ]:
# Changing kernel size to (5, 5)
model_kernel_change = models.Sequential()
model_kernel_change.add(layers.Conv2D(32, (5, 5), activation='relu', input_shape=(32, 32, 3)))
model_kernel_change.add(layers.MaxPooling2D((2, 2)))
model_kernel_change.add(layers.Conv2D(64, (5, 5), activation='relu'))
model_kernel_change.add(layers.MaxPooling2D((2, 2)))
model_kernel_change.add(layers.Conv2D(64, (5, 5), activation='relu'))
model_kernel_change.add(layers.Flatten())
model_kernel_change.add(layers.Dense(64, activation='relu'))
model_kernel_change.add(layers.Dense(10))
model_kernel_change.summary()

model_kernel_change.compile(optimizer='adam',
                            loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                            metrics=['accuracy'])

# Training with changed kernel size for 10 epochs
history_kernel_change_10_epochs = model_kernel_change.fit(train_images, train_labels, epochs=10,
                                                          validation_data=(test_images, test_labels))

plt.figure(figsize=(4, 4))
plt.plot(history_kernel_change_10_epochs.history['accuracy'], label='accuracy')
plt.plot(history_kernel_change_10_epochs.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.5, 1])
plt.title('history_kernel_change_10_epochs')
plt.legend(loc='lower right')


test_loss_kernel_change, test_acc_kernel_change = model_kernel_change.evaluate(test_images, test_labels)
print('Test accuracy (Changed Kernel Size):', test_acc_kernel_change)


Training the CNN model for different epochs showed that increasing the epochs initially improved test accuracy (0.6948 for 10 epochs), but with further increases, it led to overfitting, resulting in a decline in accuracy (0.6851 for 20 epochs and 0.6756 for 50 epochs). Changing the kernel size from (3, 3) to (5, 5) did not significantly impact accuracy (0.693).